# Algérie Télécom: Unsupervised Topic Discovery
**Objective:** Since our current dataset does not contain timestamps, we cannot map trends over time. Instead, we will use **BERTopic** to perform unsupervised cluster analysis.

**Goal:** Discover hidden, granular sub-topics within our customer feedback (e.g., separating general "negative" comments into specific clusters like "router configuration issues," "fiber installation delays," or "billing errors"). This will serve as our "Baseline Topic Dictionary" for future dynamic trend analysis when timestamped data becomes available.

## Part 1: Data Loading
We begin by combining our raw data sources (8 CSV files and 1 Excel file) into a single master DataFrame for analysis.

In [ ]:
import pandas as pd
import glob
import os
import re

data_path = './'

# 1. Grab all CSV and XLSX files
csv_files = glob.glob(os.path.join(data_path, '*.csv'))
excel_files = glob.glob(os.path.join(data_path, '*.xlsx'))

print(f"Found {len(csv_files)} CSV files and {len(excel_files)} Excel files.")

# 2. Read all files into a list of DataFrames
dataframes = []

for file in csv_files:
    dataframes.append(pd.read_csv(file))

for file in excel_files:
    dataframes.append(pd.read_excel(file))

# 3. Concatenate all DataFrames into one large DataFrame
df = pd.concat(dataframes, ignore_index=True)

# 4. Drop any exact duplicate rows that might have occurred across files
initial_shape = df.shape
df = df.drop_duplicates()
final_shape = df.shape

print(f"Data merged successfully!")
print(f"Rows before deduplication: {initial_shape[0]}")
print(f"Rows after deduplication: {final_shape[0]}")

Found 8 CSV files and 1 Excel files.
Data merged successfully!
Rows before deduplication: 9881
Rows after deduplication: 9855


## Part 2: Preparing Data for BERTopic
We will extract the `Commentaire client` column to perform a static cluster analysis.

BERTopic uses deep learning (Sentence Transformers) to understand full-sentence context, so we do not need to do heavy preprocessing (like removing stop words or stemming) before feeding it the raw Darija/French data.

In [ ]:
# Targeting the exact column from the Algérie Télécom dataset
text_column = 'Commentaire client'

# 1. Drop any rows where the customer comment is completely empty
df_clean = df.dropna(subset=[text_column])

# 2. Convert the text column into a simple list of strings (which BERTopic requires)
docs = df_clean[text_column].astype(str).tolist()

print(f"Extracted {len(docs)} valid customer comments for Topic Discovery.")

Extracted 9853 valid customer comments for Topic Discovery.


## Part 3: Running BERTopic (Multilingual)
We initialize a `SentenceTransformer` pre-trained on multiple languages so it can natively process the Algerian Darija mix (Arabic, French, and Arabizi). We then ask BERTopic to cluster these comments into hidden topics.

In [ ]:
!pip install bertopic
!pip install sentence-transformers
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# 1. Load the multilingual embedding model
print("Loading multilingual embeddings (This might take a moment)...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 2. Initialize BERTopic
# min_topic_size=20 ensures we don't get tiny, useless clusters.
topic_model = BERTopic(embedding_model=embedding_model, min_topic_size=20, language="multilingual")

# 3. Fit the model to your Darija comments
print("Clustering comments and discovering hidden topics...")
topics, probabilities = topic_model.fit_transform(docs)

# 4. View the results!
print("\n--- Top Hidden Topics Discovered ---")
topic_info = topic_model.get_topic_info()
display(topic_info.head(15))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.5 MB/s eta 0:00:00
Loading multilingual embeddings (This might take a moment)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Clustering comments and discovering hidden topics...

--- Top Hidden Topics Discovered ---


,Topic,Count,Name,Representation,Representative_Docs
0,-1,3601,-1_في_من_مودام_لا,"[في, من, مودام, لا, على, لي, المودام, ما, الزب...",[راني نستنا تحلولي مشكل تاعي من أكتوبر لي يومن...
1,0,1036,0_الجزائر_إتصالات_algérie_télécom,"[الجزائر, إتصالات, algérie, télécom, اتصالات, ...",[Algérie Télécom - إتصالات الجزائر فضاء الزبون...
2,1,911,1_والله_في_ولا_من,"[والله, في, ولا, من, انترنت, مكانش, المودام, ل...","[رانا نستناو في المودام قالك مكانش, 5 شهر و حن..."
3,2,349,2_اشهر_شهر_أشهر_يوم,"[اشهر, شهر, أشهر, يوم, منذ, شهرين, عندي, ايام,...","[لايوجد مودام فيبر دفعت الاشتراك منذ شهر, مودا..."
4,3,258,3_4g_مودام_lte_idoom,"[4g, مودام, lte, idoom, 5g, ايدوم, 4جي, 4glte,...","[نريد لي idoom 4g, هل متوفر مودام ايدوم 4g, شح..."
5,4,244,4_البصرية_الألياف_الالياف_la,"[البصرية, الألياف, الالياف, la, بالألياف, fibr...","[نريد توفير الألياف البصرية, عندكم مودام الالي..."
6,5,210,5_الانترنت_انترنت_internet_الأنترنت,"[الانترنت, انترنت, internet, الأنترنت, جدا, ال...",[في وقت وين حنا الباحثيين رانا نوجدو في كلاود ...
7,6,162,6_ولاية_بلدية_وسط_عنابة,"[ولاية, بلدية, وسط, عنابة, اين, أين, المدينة, ...","[اين هو الفيبر, احنا وسط المدينة ولاية سكيكدة ..."
8,7,145,7_العرض_عرض_العروض_هذا,"[العرض, عرض, العروض, هذا, عروض, وهمية, فيديو, ...","[سلام كيف الاستفادة من هذا العرض, السلام عليكم..."
9,8,141,8_ميغا_50_100_دج,"[ميغا, 50, 100, دج, الف, 500, 300, جيغا, 20, 15]",[ومن 15 ل 20 تزيد 60 الف بزاف مليحا ترجعونا 20...


## Part 4: Visualization
To better understand what the AI has discovered, we can visualize the top words that define the biggest complaint clusters.

In [ ]:
# This generates an interactive bar chart of the top 8 topics
fig = topic_model.visualize_barchart(top_n_topics=8)
fig.show()

### Visual Analysis of Discovered Topics
BERTopic successfully clustered the unstructured customer feedback into distinct, actionable business themes. By analyzing the top words (TF-IDF scores) in each cluster, we can assign human-readable labels to these hidden topics:

* **Topic 0 (Brand Chatter):** General mentions of the company name (*Algérie Télécom, إتصالات الجزائر*).
* **Topic 1 (Expressive Frustration):** Dialectal venting and emphasis (*والله*) regarding internet services.
* **Topic 2 (Delays & Wait Times):** Dominated by time-related words (*شهر, اشهر, يوم, منذ*), highlighting a major pain point regarding wait times for technical support or installations.
* **Topic 3 (4G/LTE Modems):** Hardware-specific feedback regarding Idoom 4G and LTE modems.
* **Topic 4 (Fiber Optics - FTTH):** Exclusively focused on Fiber deployment, requests, and optical network feedback (*الألياف البصرية*).
* **Topic 5 (Connection Quality):** General internet performance issues, heavily featuring the word *جدا* (very), strongly implying "very slow" or "very weak" connections.
* **Topic 6 (Coverage & Geography):** Inquiries or reports specific to geographic locations, municipalities, and provinces (*ولاية, بلدية, عنابة*).
* **Topic 7 (Commercial Offers):** Discussions revolving around pricing, new packages, and promotions (*عروض, العرض*).

**Strategic Business Value:** We have successfully transitioned from generic sentiment analysis to granular operational intelligence. Rather than simply reporting that "complaints are up," the Algérie Télécom moderation team can now route Topic 3 directly to the 4G Hardware team, Topic 4 to the Fiber deployment team, and Topic 7 to the Commercial/Marketing team.

## Part 5: Locking in Business Taxonomy (Naming the Clusters)
To ensure our management dashboards remain stable over time, we must lock in human-readable business names for our discovered topics.

If we rely on automated AI naming every month, a topic might be called "Router Issues" today and "Modem Problems" tomorrow, which breaks historical tracking. By defining a custom label dictionary, we create a permanent "Baseline Topic Dictionary" for Algérie Télécom.

In [ ]:
# 1. Define your custom human-readable labels based on your analysis
# Note: Topic -1 is ALWAYS the "Outlier/Noise" cluster in BERTopic
custom_labels = {
    -1: "Uncategorized / Noise",
    0: "General Brand Mentions",
    1: "Frustration & Venting",
    2: "Wait Times & Delays",
    3: "4G/LTE Modems",
    4: "Fiber Optics (FTTH)",
    5: "Slow Connection Speed",
    6: "Coverage Inquiries",
    7: "Commercial Offers"
    # Add more if BERTopic found more than 7!
}

# 2. Apply these labels to your saved model
topic_model.set_topic_labels(custom_labels)

# 3. View the charts with the proper Business Names!
fig = topic_model.visualize_barchart(top_n_topics=8, custom_labels=True)
fig.show()

## Part 6: Exporting the Baseline Master Model
With our topics professionally named and locked in, our baseline model is complete. We will now export this model.

This notebook represents our "Laboratory." From this point forward, the model will live on the backend servers. We will hand off this exported model along with two production Python scripts (`monthly_auto_updater.py` and `topic_manager.py`) to handle real-time routing, future anomaly detection, and topic renaming.

In [ ]:
import os

# Create the export directory
os.makedirs("./export/algerie_telecom_bertopic", exist_ok=True)

# Save the model
print("Saving the baseline BERTopic model...")
topic_model.save("./export/algerie_telecom_bertopic", serialization="safetensors")

print("Model successfully exported! The Handoff Package is ready.")

Saving the baseline BERTopic model...
Model successfully exported! The Handoff Package is ready.


In [ ]:
import os

# 1. Create a dedicated folder so your workspace stays clean
export_folder = "./exported_topics_review"
os.makedirs(export_folder, exist_ok=True)

# 2. Attach the AI's topic IDs and your custom names to the dataframe
# (Assuming 'df_clean' is your dataframe and 'topics' is the output from fit_transform)
df_clean['Topic_ID'] = topics
df_clean['Topic_Name'] = [topic_model.custom_labels_[t + topic_model._outliers] for t in topics]

print(f"Exporting {len(set(topics))} topics to individual Excel files...")

# 3. Loop through every unique topic ID and create a separate file
for topic_id in set(topics):

    # Filter the dataframe to isolate just the current topic's comments
    topic_data = df_clean[df_clean['Topic_ID'] == topic_id]

    # Get the custom name and format it to be a safe computer filename
    # (Replacing spaces and slashes with underscores so it doesn't break the file path)
    raw_name = topic_model.custom_labels_[topic_id + topic_model._outliers]
    safe_name = raw_name.replace("/", "_").replace("\\", "_").replace(" ", "_")

    # Create the file name (e.g., "Topic_4_Fiber_Optics_(FTTH).xlsx")
    filepath = f"{export_folder}/Topic_{topic_id}_{safe_name}.xlsx"

    # Save it to Excel! We use index=False so it doesn't export the random row numbers
    topic_data.to_excel(filepath, index=False)

print(f"✅ All done! Open the '{export_folder}' folder on the left to download and check your files.")

Exporting 55 topics to individual Excel files...
✅ All done! Open the './exported_topics_review' folder on the left to download and check your files.
